# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jawad-ahmed-developer/flyRank_Internship_Tasks/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

## Week 4 — Baseline Action Score

### Lane

**Refresh / Content Opportunity Scoring**

The decision is: **which existing content items should an SEO reviewer consider for a content refresh first?**

Week 3 established five candidate features. For this baseline, I will use two of them as the primary rule signals:

* `days_since_last_update` — content freshness / staleness
* `imp_prev30` — recent search visibility

The rule is intentionally simple and transparent. It is a decision-support baseline, not a trained model.

I will first audit both signals using bucketed comparisons and visible sample counts (`n`). I will then freeze the rule before building the ranked queue.

The rule will use only information available at the decision point. Future performance and outcome-derived fields will not be used.


In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# Week 4 setup — connect to the FlyRank warehouse

%pip -q install duckdb huggingface_hub

import os
import getpass
import duckdb
import pandas as pd
import numpy as np

# Token order:
# 1. environment variable
# 2. Colab Secret
# 3. secure prompt
HF_TOKEN = os.environ.get("HF_TOKEN")

if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get("HF_TOKEN")
    except Exception:
        pass

if not HF_TOKEN:
    HF_TOKEN = getpass.getpass(
        "Paste your Hugging Face READ token (hf_...): "
    )

con = duckdb.connect()

con.execute(
    f"""
    CREATE OR REPLACE SECRET hf
    (TYPE huggingface, TOKEN '{HF_TOKEN}')
    """
)

REL = "hf://datasets/FlyRank/internship-warehouse"

TABLES = {
    "dim_clients":
        f"read_parquet('{REL}/dim_clients.parquet')",

    "dim_content":
        f"read_parquet('{REL}/dim_content.parquet')",

    "fact_daily":
        f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",

    "fact_daily_sample":
        f"read_parquet('{REL}/fact_content_daily_performance_sample.parquet')",

    "fact_query_90d":
        f"read_parquet('{REL}/fact_content_query_90d.parquet')",
}

print("Connected to FlyRank warehouse.")
print("Development slice: March 2026")

Paste your Hugging Face READ token (hf_...): ··········
Connected to FlyRank warehouse.
Development slice: March 2026


### Inspect the content dimension

Before constructing the rule, I will inspect the available `dim_content` fields so that content age and update freshness are derived from the warehouse metadata rather than assumed. This keeps the Week 4 rule tied to the actual data contract.


In [2]:
# Inspect dim_content schema before using metadata fields

content_schema = con.sql(
    f"DESCRIBE SELECT * FROM {TABLES['dim_content']}"
).df()

display(
    content_schema[
        ["column_name", "column_type"]
    ]
)

,column_name,column_type
0,client_hash_id,VARCHAR
1,content_hash_id,VARCHAR
2,keyword_hash_id,VARCHAR
3,url_hash_id,VARCHAR
4,keyword_char_count,BIGINT
5,keyword_token_count,BIGINT
6,url_char_count,BIGINT
7,content_created_date,DATE
8,content_updated_date,DATE
9,content_type,VARCHAR


### Inspect the performance table

The Week 3 contract defines `imp_prev30`, `clicks_prev30`, and `avg_position_prev30` as decision-time features.

I will inspect the performance table schema before constructing these features from the full release. This avoids assuming column names and keeps the baseline reproducible against the actual FlyRank warehouse.


In [3]:
# Inspect the daily performance schema

performance_schema = con.sql(
    f"DESCRIBE SELECT * FROM {TABLES['fact_daily_sample']}"
).df()

display(
    performance_schema[
        ["column_name", "column_type"]
    ]
)

,column_name,column_type
0,report_date,DATE
1,client_hash_id,VARCHAR
2,content_hash_id,VARCHAR
3,client_has_gsc,BOOLEAN
4,client_has_ga4,BOOLEAN
5,gsc_data_available,BOOLEAN
6,ga4_data_available,BOOLEAN
7,gsc_impressions,BIGINT
8,gsc_clicks,BIGINT
9,gsc_sum_position,BIGINT


### Build the decision-time feature table

For the baseline, I use only information that would have been available when the decision was made.

For each content item and decision date:

* `imp_prev30` = total Google Search Console impressions during the previous 30 days
* `clicks_prev30` = total Google Search Console clicks during the previous 30 days
* `avg_position_prev30` = impression-weighted average position during the previous 30 days
* `content_age_days` = days since the content was created
* `days_since_last_update` = days since the content was last updated

The baseline rule will use `imp_prev30` and `days_since_last_update`. The other three contracted features remain available but are not required for this simple rule.

No future-period performance or outcome-derived feature is used.


In [6]:
# 1D — Build the Week 3 decision-time feature table
# Development window: March 2026
# June 2026 sample remains sealed for final evaluation.

DECISION_START = "2026-03-01"
DECISION_END   = "2026-03-31"

FACT_DAILY = """
read_parquet(
    'hf://datasets/FlyRank/internship-warehouse/
     fact_content_daily_performance/**/*.parquet',
    hive_partitioning=true
)
"""

feature_sql = f"""
WITH decision_rows AS (
    SELECT DISTINCT
        report_date,
        client_hash_id,
        content_hash_id
    FROM {FACT_DAILY}
    WHERE report_date BETWEEN DATE '{DECISION_START}'
                          AND DATE '{DECISION_END}'
      AND gsc_data_available = TRUE
),

prev30 AS (
    SELECT
        d.report_date,
        d.client_hash_id,
        d.content_hash_id,

        SUM(p.gsc_impressions) AS imp_prev30,
        SUM(p.gsc_clicks) AS clicks_prev30,

        CASE
            WHEN SUM(p.gsc_impressions) > 0
            THEN SUM(
                p.gsc_avg_position * p.gsc_impressions
            ) / SUM(p.gsc_impressions)
            ELSE NULL
        END AS avg_position_prev30

    FROM decision_rows d

    LEFT JOIN {FACT_DAILY} p
      ON p.client_hash_id = d.client_hash_id
     AND p.content_hash_id = d.content_hash_id
     AND p.report_date >= d.report_date - INTERVAL 30 DAY
     AND p.report_date < d.report_date
     AND p.gsc_data_available = TRUE

    GROUP BY
        d.report_date,
        d.client_hash_id,
        d.content_hash_id
)

SELECT
    p.report_date,
    p.client_hash_id,
    p.content_hash_id,

    p.imp_prev30,
    p.clicks_prev30,
    p.avg_position_prev30,

    DATE_DIFF(
        'day',
        c.content_created_date,
        p.report_date
    ) AS content_age_days,

    DATE_DIFF(
        'day',
        c.content_updated_date,
        p.report_date
    ) AS days_since_last_update

FROM prev30 p

INNER JOIN {TABLES["dim_content"]} c
    ON c.client_hash_id = p.client_hash_id
   AND c.content_hash_id = p.content_hash_id

WHERE p.imp_prev30 IS NOT NULL
  AND p.clicks_prev30 IS NOT NULL
  AND p.avg_position_prev30 IS NOT NULL
  AND c.content_created_date IS NOT NULL
  AND c.content_updated_date IS NOT NULL
  AND c.is_published = TRUE
  AND c.is_deleted = FALSE
"""

df = con.sql(feature_sql).df()

print("Rows:", len(df))
print("Unique content:", df["content_hash_id"].nunique())
print(
    "Decision dates:",
    df["report_date"].min(),
    "to",
    df["report_date"].max()
)

display(df.head())

HTTPException: HTTP Error: HTTP GET error on 'https://huggingface.co/api/datasets/FlyRank/internship-warehouse/tree/main/
     fact_content_daily_performance' (HTTP 404)

In [7]:
# Check the actual files available under the dataset root
files = con.sql("""
    SELECT *
    FROM glob(
        'hf://datasets/FlyRank/internship-warehouse/*'
    )
    LIMIT 30
""").df()

display(files)

,file
0,hf://datasets/FlyRank/internship-warehouse/.gi...
1,hf://datasets/FlyRank/internship-warehouse/REA...
2,hf://datasets/FlyRank/internship-warehouse/dim...
3,hf://datasets/FlyRank/internship-warehouse/dim...
4,hf://datasets/FlyRank/internship-warehouse/fac...
5,hf://datasets/FlyRank/internship-warehouse/fac...


In [8]:
files = con.sql("""
    SELECT file
    FROM glob(
        'hf://datasets/FlyRank/internship-warehouse/*'
    )
    WHERE file LIKE '%fact%'
""").df()

display(files)

,file
0,hf://datasets/FlyRank/internship-warehouse/fac...
1,hf://datasets/FlyRank/internship-warehouse/fac...


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.